<a href="https://colab.research.google.com/github/HuchpechTW/public-colabs/blob/main/SPICEUP_SearchTerms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import auth
from google.auth import default
import gspread
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

# ==========================================
# 1. 配置参数 (你可以根据需求调整)
# ==========================================
COST_THRESHOLD = 15.0  # 累计消耗超过多少且 0 转化时触发报警
INFO_WORDS = ["review", "test", "how to", "legit", "meaning", "wikipedia", "youtube", "reddit"]
LOW_INTENT_WORDS = ["free", "cheap", "diy", "wholesale", "bulk", "clearance", "used", "refurbished"]
SUPPORT_WORDS = ["return", "customer service", "instruction", "manual", "warranty", "login", "contact"]
BRAND_RISK_WORDS = ["nike", "supreme", "amazon", "casetify", "ebay", "walmart", "temu", "shein", "ali express"]

# ==========================================
# 2. 验证你的账号权限（运行会弹窗让你点授权）
# ==========================================
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

search_term_excel='Search-terms-report-202602'

worksheet = gc.open(search_term_excel).sheet1
all_rows = worksheet.get_all_values()
header = all_rows[2]   # 获取第 3 行作为表头
data = all_rows[3:]    # 获取第 4 行及以后的所有数据

df = pd.DataFrame(data, columns=header)
print(df.head())

# 清理数据：移除合计行，转换数值类型
df = df[df['Search term'].notnull()]
df = df[~df['Search term'].str.contains('Total:', na=False)]
df['Cost'] = pd.to_numeric(df['Cost'], errors='coerce').fillna(0)
df['Conversions'] = pd.to_numeric(df['Conversions'], errors='coerce').fillna(0)
df['Clicks'] = pd.to_numeric(df['Clicks'], errors='coerce').fillna(0)
df['Impr.'] = pd.to_numeric(df['Impr.'], errors='coerce').fillna(0)

# ==========================================
# 3. N-Gram 分析逻辑
# ==========================================
def get_ngrams(text, n):
    if not isinstance(text, str): return []
    tokens = re.findall(r'\b\w+\b', text.lower())
    if len(tokens) < n: return []
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

ngram_stats = defaultdict(lambda: {'Cost': 0.0, 'Conv': 0.0, 'Clicks': 0, 'Impr': 0})

print("正在进行 N-Gram 聚合计算...")
for _, row in df.iterrows():
    term = row['Search term']
    for n in [1, 2, 3]:
        grams = get_ngrams(term, n)
        for gram in grams:
            ngram_stats[gram]['Cost'] += row['Cost']
            ngram_stats[gram]['Conv'] += row['Conversions']
            ngram_stats[gram]['Clicks'] += row['Clicks']
            ngram_stats[gram]['Impr'] += row['Impr.']

# 转为 DataFrame 并计算指标
ngram_df = pd.DataFrame.from_dict(ngram_stats, orient='index').reset_index()
ngram_df.rename(columns={'index': 'ngram'}, inplace=True)
ngram_df['CPA'] = np.where(ngram_df['Conv'] > 0, ngram_df['Cost'] / ngram_df['Conv'], np.inf)

# ==========================================
# 4. 提取否定词建议
# ==========================================
suggestions = []

# 策略 A: 高损耗零转化词
high_waste = ngram_df[(ngram_df['Cost'] >= COST_THRESHOLD) & (ngram_df['Conv'] == 0)]
for _, row in high_waste.sort_values(by='Cost', ascending=False).iterrows():
    suggestions.append([f'"{row["ngram"]}"', 'Phrase', f"Cost: ${row['Cost']:.2f}, Conv: 0", 'N-Gram 高损耗且零转化'])

# 策略 B: 意图过滤 (根据预设词表)
def check_intent(word_list, reason):
    pattern = '|'.join([rf'\b{w}\b' for w in word_list])
    matched = ngram_df[ngram_df['ngram'].str.contains(pattern, na=False, case=False) & (ngram_df['Cost'] > 0)]
    for _, row in matched.iterrows():
        suggestions.append([f'"{row["ngram"]}"', 'Phrase', f"Cost: ${row['Cost']:.2f}, Conv: {row['Conv']}", reason])

check_intent(INFO_WORDS, '信息检索类意图 (调研阶段)')
check_intent(LOW_INTENT_WORDS, '低端意图 (客单价不匹配)')
check_intent(SUPPORT_WORDS, '售后/支持类 (非购买意图)')
check_intent(BRAND_RISK_WORDS, '版权风险或竞品干扰')

# ==========================================
# 5. 输出结果
# ==========================================
final_report = pd.DataFrame(suggestions, columns=['Negative Keyword', 'Match Type', 'Data Evidence', 'Strategic Reason'])
final_report.drop_duplicates(subset=['Negative Keyword'], inplace=True)

print("\n--- 分析完成！建议排除以下关键词 ---")
display(final_report.sort_values(by='Data Evidence', ascending=False))

# 导出到 CSV
final_report.to_csv('negative_keyword_suggestions.csv', index=False)
print("\n建议结果已保存至: negative_keyword_suggestions.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
                     Search term       Match type Added/Excluded  \
0      green cover for iphone 13  Performance Max           None   
1  aesthetic phone cases for men  Performance Max           None   
2                       dc chain  Performance Max           None   
3              drip iphone cases  Performance Max           None   
4            airpods pro 2 coque  Performance Max           None   

                 Campaign Ad group Currency code Avg. CPM Clicks Impr. CTR  \
0  PMax-25May28-Main-R150       --           USD        0      0     1  0%   
1  PMax-25May28-Main-R150       --           USD        0      0     5  0%   
2  PMax-25May28-Main-R150       --           USD        0      0     2  0%   
3  PMax-25May28-Main-R150       --           USD        0      0     2  0%   
4  PMax-25May28-Main-R150       --           USD        0      0     1  

,Negative Keyword,Match Type,Data Evidence,Strategic Reason
77,"""supreme case""",Phrase,"Cost: $9.51, Conv: 1.0",版权风险或竞品干扰
273,"""17 case nike""",Phrase,"Cost: $9.12, Conv: 0.0",版权风险或竞品干扰
0,"""silicone iphone""",Phrase,"Cost: $87.24, Conv: 0",N-Gram 高损耗且零转化
150,"""max case nike""",Phrase,"Cost: $8.99, Conv: 2.0",版权风险或竞品干扰
191,"""max case supreme""",Phrase,"Cost: $8.75, Conv: 0.0",版权风险或竞品干扰
...,...,...,...,...
58,"""how to turn""",Phrase,"Cost: $0.03, Conv: 0.0",信息检索类意图 (调研阶段)
59,"""how to stop""",Phrase,"Cost: $0.01, Conv: 0.0",信息检索类意图 (调研阶段)
228,"""nike astronaut phone""",Phrase,"Cost: $0.01, Conv: 0.0",版权风险或竞品干扰
227,"""nike astronaut""",Phrase,"Cost: $0.01, Conv: 0.0",版权风险或竞品干扰



建议结果已保存至: negative_keyword_suggestions.csv
